# Verify DMR-IR Dataset Labels
This notebook installs the required dependencies, loads the `SemilleroCV/DMR-IR` dataset metadata, and extracts all the `ClassLabel` features to verify the available classes.

In [1]:
# Install datasets library if not present
!pip install datasets huggingface_hub

  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 4.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 693.4/693.4 kB 6.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 32.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 24.8 MB/s  0:00:00
Using cached h11-0.16.0-py3-none-any.whl (37 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.8/48.8 MB 60.0 MB/s  0:00:006m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 806.6/806.6 kB 10.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23/23 [datasets]/23 [datasets]ce_hub]


In [6]:
from datasets import load_dataset, ClassLabel

# NOTE: If the dataset is gated, you'll need to login to Hugging Face.
# Uncomment the lines below to login interactively if you aren't already logged in via CLI.
# from huggingface_hub import notebook_login
# notebook_login()

# We use streaming=True so it doesn't download the entire 5GB dataset right away,
# but instead just fetches the dataset info/metadata to check the features.
print("Loading dataset metadata...")
ds = load_dataset("SemilleroCV/DMR-IR", split="train", streaming=True)
print("Dataset metadata loaded successfully!")

Loading dataset metadata...
Dataset metadata loaded successfully!


In [5]:
print("=== DMR-IR Categorical Labels & Features ===\n")
features = ds.features

for feature_name, feature_type in features.items():
    if isinstance(feature_type, ClassLabel):
        print(f"Feature: '{feature_name}'")
        print("-" * 40)
        for idx, label_name in enumerate(feature_type.names):
            print(f"  {idx}: {label_name}")
        print("\n")

=== DMR-IR Categorical Labels & Features ===

Feature: 'label'
----------------------------------------
  0: benign
  1: malignant


Feature: 'view'
----------------------------------------
  0: Frontal
  1: Right 45°
  2: Right 90°
  3: Left 45°
  4: Left 90°
  5: Unknown


Feature: 'marital_status'
----------------------------------------
  0: Divorced
  1: Married
  2: Single
  3: Widow


Feature: 'race'
----------------------------------------
  0: Asian
  1: Black
  2: Indigenous
  3: Mulatto
  4: Multiracial
  5: White


Feature: 'eating_habits'
----------------------------------------
  0: 35.50
  1: High in fat
  2: Low in fat
  3: No fat


Feature: 'mammography'
----------------------------------------
  0: No
  1: Yes


Feature: 'radiotherapy'
----------------------------------------
  0: No
  1: Yes


Feature: 'plastic_surgery'
----------------------------------------
  0: No
  1: Yes, both breasts
  2: Yes, left breast
  3: Yes, right breast


Feature: 'prosthesis'
--------

In [9]:
categorical_features = [
    name for name, feature in info.features.items()
    if hasattr(feature, "names") and name != "label"
]

summary_rows = []
for feature_name in categorical_features:
    feature = info.features[feature_name]
    counts = Counter()
    train_stream = load_dataset_builder("SemilleroCV/DMR-IR").as_streaming_dataset(split="train")
    for example in train_stream:
        value = example[feature_name]
        if isinstance(value, int) and value < len(feature.names):
            value = feature.names[value]
        counts[str(value)] += 1

    for class_name, count in counts.items():
        summary_rows.append({
            "feature": feature_name,
            "class": class_name,
            "count": count,
            "percent": count / sum(counts.values()) * 100,
        })

summary_df = pd.DataFrame(summary_rows)
display(summary_df.sort_values(["feature", "count"], ascending=[True, False]))

if not summary_df.empty:
    n_features = len(categorical_features)
    n_cols = 2
    n_rows = math.ceil(n_features / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, max(4, 4 * n_rows)))
    axes = axes.flatten() if hasattr(axes, "flatten") else [axes]

    for ax, feature_name in zip(axes, categorical_features):
        feature_data = summary_df[summary_df["feature"] == feature_name].sort_values("count", ascending=False)
        sns.barplot(data=feature_data, x="count", y="class", ax=ax, palette="mako")
        ax.set_title(feature_name)
        ax.set_xlabel("Count")
        ax.set_ylabel("")

    for ax in axes[len(categorical_features):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

NameError: name 'info' is not defined

In [ ]:
print("=== Label Distribution ===")

label_counts = Counter()
label_feature = info.features["label"]
label_names = list(label_feature.names)

# Stream through the training split so we count every sample without loading the full dataset into memory.
train_stream = load_dataset_builder("SemilleroCV/DMR-IR").as_streaming_dataset(split="train")
for example in train_stream:
    label_value = example["label"]
    if isinstance(label_value, int):
        label_name = label_names[label_value]
    else:
        label_name = str(label_value)
    label_counts[label_name] += 1

label_df = (
    pd.DataFrame(label_counts.items(), columns=["label", "count"])
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)
label_df["percent"] = label_df["count"] / label_df["count"].sum() * 100
display(label_df)

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=label_df, x="label", y="count", palette="viridis", ax=ax)
ax.set_title("Class Distribution: Benign vs Malignant")
ax.set_xlabel("Class")
ax.set_ylabel("Number of images")
for container in ax.containers:
    ax.bar_label(container, fmt="%d", padding=3)
plt.tight_layout()
plt.show()

=== Label Distribution ===


In [13]:
print("=== Full Dataset Metadata Summary ===\n")

builder = load_dataset_builder("SemilleroCV/DMR-IR")
info = builder.info

split_rows = []
for split_name, split_info in info.splits.items():
    split_rows.append({
        "split": split_name,
        "num_examples": split_info.num_examples,
        "num_bytes": split_info.num_bytes,
    })

metadata_df = pd.DataFrame(split_rows)
if not metadata_df.empty:
    display(metadata_df)

if "train" in info.splits and info.splits["train"].num_examples is not None:
    total_images = info.splits["train"].num_examples
else:
    total_images = sum(split_info.num_examples or 0 for split_info in info.splits.values())

print(f"Total images in dataset: {total_images:,}")

feature_rows = []
for feature_name, feature_type in info.features.items():
    feature_rows.append({
        "feature": feature_name,
        "type": type(feature_type).__name__,
        "num_classes": len(feature_type.names) if hasattr(feature_type, "names") else None,
        "classes": ", ".join(feature_type.names) if hasattr(feature_type, "names") else None,
    })

feature_df = pd.DataFrame(feature_rows)
display(feature_df)

=== Full Dataset Metadata Summary ===



,split,num_examples,num_bytes
0,train,4874,6.262191e+09
1,validation,980,1.259102e+09
2,test,1024,1.315635e+09


Total images in dataset: 4,874


,feature,type,num_classes,classes
0,image,Image,NaN,NaN
1,label,ClassLabel,2.0,"benign, malignant"
2,text,Value,NaN,NaN
3,patient_id,Value,NaN,NaN
4,text_embedding,List,NaN,NaN
5,segmentation_mask,List,NaN,NaN
6,protocol,Value,NaN,NaN
7,view,ClassLabel,6.0,"Frontal, Right 45°, Right 90°, Left 45°, Left ..."
8,record,Value,NaN,NaN
9,role,Value,NaN,NaN


In [12]:
from collections import Counter
import math

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from datasets import load_dataset_builder

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

## Dataset Metadata Overview
This section summarizes the dataset structure, total image count, label balance, and the distribution of the remaining categorical metadata fields.